Task 2
Scrape job listings from the website https://realpython.github.io/fake-jobs and store the data into an SQLite database.

1. Scraping Requirements:

Extract the following details for each job listing:
Job Title
Company Name
Location
Job Description
Application Link

2. Data Storage:

Store the scraped data into an SQLite database in a table named jobs.

3. Incremental Load:

Ensure that your script performs incremental loading:
Scrape the webpage and add only new job listings to the database.
Avoid duplicating entries. Use Job Title, Company Name, and Location as unique identifiers for comparison.

4. Update Tracking:

Add functionality to detect if an existing job listing has been updated (e.g., description or application link changes) and update the database record accordingly.

5. Filtering and Exporting:

Allow filtering job listings by location or company name.
Write a function to export filtered results into a CSV file.

In [8]:
from bs4 import BeautifulSoup
import requests
url = "https://realpython.github.io/fake-jobs"
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

jobs = soup.find_all("div", class_ = "card-content")
job_list= []
for job in jobs:
    title = job.find("h2", class_ = "title")
    company = job.find("h3", class_ = "subtitle")
    location = job.find("p", class_ = "location")
    link = job.find("a", string = "Apply")["href"]

    job_page = requests.get(link)
    desc = BeautifulSoup(job_page.text, "html.parser")
    description = desc.find("div", class_ = "content").find("p").text

    job_list.append({
        "title" : title.text,
        "company" : company.text,
        "location" : location.text.strip(),
        "description" : description,
        "link" : link   
    })



In [9]:
import sqlite3

with sqlite3.connect("jobs.db") as c:
    cursor = c.cursor()
    cursor.execute("""
    CREATE TABLE IF NOT EXISTS "Jobs"(
        "title" TEXT,
        "company" TEXT,
        "location" TEXT,
        "description" TEXT,
        "link" TEXT
    )  
""")
    c.commit()

    for job in job_list:
        cursor.execute("SELECT * FROM Jobs WHERE title = ? AND company = ? AND location = ?", (job["title"], job["company"], job["location"]))

        existing = cursor.fetchone()

        if existing is None:
            cursor.execute("INSERT INTO Jobs VALUES (?, ?, ?, ?, ?)", (job["title"], job["company"], job["location"], job["description"], job["link"]))
        else:
            cursor.execute("UPDATE Jobs SET description = ?, link = ? WHERE title =? AND company = ? AND location = ?", 
                           (job["description"], job["link"], job["title"], job["company"], job["location"]))

    c.commit()

In [10]:
def filtering_jobs(location = None,company = None):
    import sqlite3
    with sqlite3.connect("jobs.db") as c:
        cursor = c.cursor()

        if location:
            cursor.execute("SELECT * FROM Jobs WHERE location = ?", (location,))
        elif company:
            cursor.execute("SELECT * FROM Jobs WHERE company = ?", (company,))
        else:
            cursor.execute("SELECT * FROM Jobs")
        result = cursor.fetchall()
        return result

filtering_jobs(location = "Hullview, AA")
    

[('Sports therapist',
  'Spencer and Sons',
  'Hullview, AA',
  'Apply western feeling. Popular too high inside somebody analysis. And give source share budget ready. Theory moment better. Senior alone option in evidence board business.',
  'https://realpython.github.io/fake-jobs/jobs/sports-therapist-82.html')]

In [13]:
def convert_to_csv(jobs, filename):
    import csv
    with open(filename, 'w', newline = '') as c:
        writer = csv.writer(c)
        writer.writerow(["Title", "Company", "Location", "Description", "Link"])
        for job in jobs:
            writer.writerow(job)
        print(f"Exported {len(jobs)} job(s) to {filename}")

        

In [14]:
jobs = filtering_jobs(location = "Hullview, AA")
convert_to_csv(jobs, 'jobs.csv')

Exported 1 job(s) to jobs.csv
